# Gold – Olist
Sterschema voor analyses: dimensies (wie, wat, waar, wanneer) en een feitentabel (verkopen).

In [0]:
from pyspark.sql import functions as F

CATALOG = "bazar9_dbw"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

def read_silver(name):
    """Lees een silver-tabel."""
    return spark.table(f"{SILVER}.{name}")

def save_gold(df, name):
    """Sla op als gold-tabel en toon het aantal rijen."""
    (
        df.withColumn("_processed_at", F.current_timestamp())
        .write.mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{GOLD}.{name}")
    )
    print(f"{name}: {spark.table(f'{GOLD}.{name}').count():,} rijen")

In [0]:
geo = read_silver("geolocation").select("zip_code_prefix", "lat", "lng")

dim_customer = (
    read_silver("customers")
    .join(geo, "zip_code_prefix", "left")
    .select(
        "customer_id",
        "customer_unique_id",
        "city",
        "state",
        "zip_code_prefix",
        "lat",
        "lng",
    )
)
save_gold(dim_customer, "dim_customer")

In [0]:
dim_product = (
    read_silver("products")
    .select(
        "product_id",
        "category",
        "photos_qty",
        "weight_g",
        "length_cm",
        "height_cm",
        "width_cm",
        (F.col("length_cm") * F.col("height_cm") * F.col("width_cm")).alias("volume_cm3"),
    )
)
save_gold(dim_product, "dim_product")

dim_seller = (
    read_silver("sellers")
    .join(geo, "zip_code_prefix", "left")
    .select(
        "seller_id",
        "city",
        "state",
        "zip_code_prefix",
        "lat",
        "lng",
    )
)
save_gold(dim_seller, "dim_seller")

In [0]:
bounds = read_silver("orders").agg(
    F.min(F.to_date("purchased_at")).alias("start_date"),
    F.greatest(
        F.max(F.to_date("estimated_delivery_at")),
        F.max(F.to_date("delivered_customer_at")),
    ).alias("end_date"),
)

dim_date = (
    bounds
    .select(F.explode(F.sequence("start_date", "end_date")).alias("date"))
    .select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        "date",
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.weekofyear("date").alias("week"),
        F.dayofmonth("date").alias("day"),
        F.date_format("date", "EEEE").alias("day_name"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
    )
)
save_gold(dim_date, "dim_date")

In [0]:
orders = read_silver("orders").select(
    "order_id",
    "customer_id",
    "order_status",
    "purchased_at",
    "delivered_customer_at",
    "estimated_delivery_at",
)

reviews_per_order = (
    read_silver("order_reviews")
    .groupBy("order_id")
    .agg(F.avg("review_score").alias("review_score"))
)

fact_sales = (
    read_silver("order_items")
    .join(orders, "order_id", "inner")
    .join(reviews_per_order, "order_id", "left")
    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "product_id",
        "seller_id",
        F.date_format("purchased_at", "yyyyMMdd").cast("int").alias("order_date_key"),
        "order_status",
        "price",
        "freight_value",
        (F.col("price") + F.col("freight_value")).alias("item_total"),
        F.datediff("delivered_customer_at", "purchased_at").alias("delivery_days"),
        (F.to_date("delivered_customer_at") > F.to_date("estimated_delivery_at")).alias("is_late"),
        F.round("review_score", 1).alias("review_score"),
    )
)
save_gold(fact_sales, "fact_sales")

In [0]:
%sql

CREATE OR REPLACE TABLE bazar9_dbw.gold.sales_monthly_by_category AS
SELECT
  d.year_month,
  p.category,
  ROUND(SUM(f.item_total), 2)                 AS revenue,
  COUNT(DISTINCT f.order_id)                  AS orders,
  COUNT(*)                                    AS items_sold,
  ROUND(AVG(f.review_score), 2)               AS avg_review,
  ROUND(AVG(CAST(f.is_late AS INT)) * 100, 1) AS late_pct
FROM bazar9_dbw.gold.fact_sales f
JOIN bazar9_dbw.gold.dim_date d    ON f.order_date_key = d.date_key
JOIN bazar9_dbw.gold.dim_product p ON f.product_id = p.product_id
WHERE f.order_status = 'delivered'
GROUP BY d.year_month, p.category;

In [0]:
%sql
SELECT
  CASE WHEN is_late THEN 'te laat' ELSE 'op tijd' END AS levering,
  COUNT(DISTINCT order_id)    AS orders,
  ROUND(AVG(review_score), 2) AS gem_review
FROM bazar9_dbw.gold.fact_sales
WHERE order_status = 'delivered'
  AND is_late IS NOT NULL
GROUP BY 1;